# 02 — Preparación y ETL del dataset

**Proyecto:** Análisis de riesgo crediticio · South German Credit (UCI, id 573)  
**Alumno:** Alexander Domínguez Zárate · UTTT  
**Objetivo:** decodificar los códigos de `codetable.txt`, derivar una variable de carga financiera,
validar la calidad del resultado y generar el dataset procesado que alimentará los modelos (Unidad III)
y el Data Warehouse/Data Mart.

**Fuente de verdad del pipeline:** `src/etl/etl_south_german_credit.py`. Este notebook **reutiliza sus
mapeos y su función de validación** (sin duplicar la lógica) y documenta cada paso con interpretación.


In [1]:
import json
import os
import sys

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

# Reutilizar el pipeline canonico del ETL
sys.path.insert(0, os.path.abspath('../src/etl'))
from etl_south_german_credit import CODIGOS_VALIDOS, MAPPINGS, validate

print('mapeos de decodificacion disponibles:', len(MAPPINGS))


mapeos de decodificacion disponibles: 18


## 1) Extracción — carga del dataset crudo

El archivo `SouthGermanCredit.asc` está delimitado por espacios; la ruta es relativa a la carpeta
`notebooks/`, que es el directorio de trabajo de este notebook.


In [2]:
df = pd.read_csv('../data/raw/SouthGermanCredit.asc', sep=' ')
df.shape


(1000, 21)

In [3]:
df.head()


,laufkont,laufzeit,moral,verw,hoehe,sparkont,beszeit,rate,famges,buerge,wohnzeit,verm,alter,weitkred,wohn,bishkred,beruf,pers,telef,gastarb,kredit
0,1,18,4,2,1049,1,2,4,2,1,4,2,21,3,1,1,3,2,1,2,1
1,1,9,4,0,2799,1,3,2,3,1,2,1,36,3,1,2,3,1,1,2,1
2,2,12,2,9,841,2,4,2,2,1,4,1,23,3,1,1,2,2,1,2,1
3,1,12,4,0,2122,1,3,3,3,1,2,1,39,3,1,2,2,1,1,1,1
4,1,12,4,0,2171,1,3,4,3,1,4,2,38,1,2,2,2,2,1,1,1


**Interpretación de la extracción.**

`(1000, 21)`: se conservan los 1000 créditos y las 21 variables crudas publicadas por UCI. El `head()`
confirma que los datos llegan **codificados en enteros** sin texto ni unidades: el ETL es la etapa que los
hace legibles. Recordar del diagnóstico de la Unidad II que la muestra está ordenada por la clase objetivo,
no por fecha, por lo que el orden del archivo no tiene significado temporal.


## 2) Transformación — decodificación `*_cat`

Se aplica el diccionario de `codetable.txt` (18 variables categóricas/ordinales/binarias del catálogo).
Se agregan columnas `*_cat` **sin borrar** las columnas de códigos originales: se conserva la trazabilidad
y los códigos sirven como entrada compacta para los modelos.


In [4]:
df_dec = df.copy()
for col, mapeo in MAPPINGS.items():
    df_dec[col + '_cat'] = df_dec[col].map(mapeo)
print('columnas despues de decodificar:', df_dec.shape[1])
df_dec.filter(like='_cat').head()


columnas despues de decodificar: 39


,laufkont_cat,moral_cat,verw_cat,sparkont_cat,beszeit_cat,rate_cat,famges_cat,buerge_cat,wohnzeit_cat,verm_cat,weitkred_cat,wohn_cat,bishkred_cat,beruf_cat,pers_cat,telef_cat,gastarb_cat,kredit_cat
0,no_checking,all_bank_paid,car_used,unknown,lt1,lt20,female_nonsingle_or_male_single,none,ge7,car_other,none,free,1,skilled_official,0_to_2,no,no,good
1,no_checking,all_bank_paid,others,unknown,1_to_4,25_to_35,male_married_widowed,none,1_to_4,unknown,none,free,2_3,skilled_official,ge3,no,no,good
2,lt0,no_credits_or_paid,retraining,lt100,4_to_7,25_to_35,female_nonsingle_or_male_single,none,ge7,unknown,none,free,1,unskilled_resident,0_to_2,no,no,good
3,no_checking,all_bank_paid,others,unknown,1_to_4,20_to_25,male_married_widowed,none,1_to_4,unknown,none,free,2_3,unskilled_resident,ge3,no,yes,good
4,no_checking,all_bank_paid,others,unknown,1_to_4,lt20,male_married_widowed,none,ge7,car_other,bank,rent,2_3,unskilled_resident,0_to_2,no,yes,good


**Interpretación de la decodificación.**

Se agregaron **18 columnas `*_cat`** (21 + 18 = 39 columnas): las numéricas reales (`laufzeit`, `hoehe`,
`alter`) no se decodifican porque ya son valores medidos. Las `*_cat` traducen cada código a su etiqueta
(`laufkont = 4 → ge200_or_salary`, `moral = 3 → existing_paid`, …).

Dos decisiones de diseño documentadas:
- **No sobrescribir los códigos originales**: los modelos pueden usar los códigos (más compactos y
ordenables en ordinales) y el Data Warehouse puede usar las etiquetas (legibles en el dashboard).
- **Mapeos en el código, no en el CSV**: la fuente de verdad es `MAPPINGS` en el script ETL, derivado de
`codetable.txt`; así cualquier reproceso produce los mismos resultados.


## 3) Variable derivada — cuota aproximada mensual

`cuota_aprox_mensual = hoehe / laufzeit` (monto sobre duración, redondeo a 2 decimales). Es un proxy de la
carga mensual del crédito; **no** es la cuota real del contrato, que depende de `rate` (porcentaje del
ingreso disponible) y de la estructura de pagos de 1973–1975.


In [5]:
df_dec['cuota_aprox_mensual'] = (df_dec['hoehe'] / df_dec['laufzeit']).round(2)
df_dec[['hoehe', 'laufzeit', 'cuota_aprox_mensual']].head(8)
print(df_dec['cuota_aprox_mensual'].describe().round(2).to_string())


count    1000.00
mean      167.69
std       153.49
min        24.06
25%        89.33
50%       130.33
75%       206.18
max      2482.67


**Interpretación de la variable derivada.**

La cuota aproximada promedia **167.69 DM/mes** (mediana 130.33), con un rango de 24.06 a 2482.67 y rango
intercuartílico 89.33–206.18. El máximo (2482.67) corresponde a un crédito grande de plazo corto: es un
cliente con carga mensual extrema, exactamente el tipo de caso que el dashboard deberá destacar. La
distribución es sesgada a la derecha, herencia del sesgo de `hoehe`. En el Data Warehouse esta variable ya
se calcula también en SQL (vista `dm_riesgo_crediticio`), lo que permite validar que ETL y DW coinciden.


## 4) Validaciones de calidad post-ETL

Se ejecuta `validate()` del script ETL, que reproduce los chequeos del diagnóstico de la Unidad II
(secciones 5.1–5.3) ahora sobre el dataset **transformado**, no solo el crudo.


In [6]:
reporte = validate(df, df_dec)
for chequeo in reporte['cheques']:
    estado = 'OK  ' if chequeo['ok'] else 'FALLA'
    print(f'[{estado}] {chequeo["check"]}: {chequeo["detalle"]}')
print('todo_ok:', reporte['todo_ok'])


[OK  ] filas_conservadas: raw=1000 dec=1000 esperado=1000
[OK  ] sin_nulos: total_nulos=0
[OK  ] sin_duplicados: duplicados=0
[OK  ] codigos_en_catalogo: columnas_con_codigos_invalidos=ninguna
[OK  ] decodificacion_completa: columnas_sin_decodificar=ninguna
[OK  ] cuota_aprox_consistente: cuota_aprox_mensual = hoehe / laufzeit (redondeo 2 decimales)
[OK  ] rangos_logicos_ok: {'laufzeit': True, 'hoehe': True, 'alter': True}
todo_ok: True


**Interpretación de las validaciones.**

Los 7 chequeos resultan **OK**, y cada uno responde a un riesgo distinto del ETL:

1. **Filas conservadas** (1000 = 1000): ninguna fila se perdió ni se duplicó en el proceso.
2. **Sin nulos**: la decodificación y la derivada no introdujeron faltantes.
3. **Sin duplicados**: el ETL no generó filas repetidas (riesgo típico de un `merge` mal especificado).
4. **Códigos en catálogo**: ninguna columna cruda tiene valores fuera de `codetable.txt`.
5. **Decodificación completa**: ninguna `*_cat` quedó con `NaN` — si un código nuevo apareciera, este
cheque sería el primero en alertar (es la defensa que sustituye a la imputación, como se documentó en U2).
6. **Cuota consistente**: la derivada coincide con el recálculo (sin errores de redondeo).
7. **Rangos lógicos**: plazos 1–120 meses, montos > 0, edades 18–100.

Este es el **contrato de calidad** del dataset procesado: si una fase posterior lo rompe, el reporte
`reporte_calidad_etl.json` muestra exactamente qué cheque falló.


## 5) Carga — persistencia del dataset procesado

Se guardan tres artefactos en `data/processed/`: CSV (interoperable, para notebooks y SQL), Parquet
(columnar y comprimido, para recargas rápidas) y el reporte de calidad en JSON (evidencia auditable).


In [7]:
os.makedirs('../data/processed', exist_ok=True)
df_dec.to_csv('../data/processed/south_german_credit_processed.csv', index=False)
df_dec.to_parquet('../data/processed/south_german_credit_processed.parquet', index=False)
reporte['shape_salida'] = list(df_dec.shape)
reporte['columnas_nuevas'] = sorted(set(df_dec.columns) - set(df.columns))
with open('../data/processed/reporte_calidad_etl.json', 'w', encoding='utf-8') as f:
    json.dump(reporte, f, ensure_ascii=False, indent=2)
print('archivos generados: CSV, Parquet y reporte_calidad_etl.json')


archivos generados: CSV, Parquet y reporte_calidad_etl.json


**Nota de reproducción.** El mismo resultado se obtiene desde la raíz del repositorio con:

```bash
py src/etl/etl_south_german_credit.py
```

(El script usa rutas relativas a la raíz; este notebook usa rutas relativas a `notebooks/`.)


## 6) Conclusión de la preparación

El dataset procesado queda en **1000 × 40** (21 crudas + 18 `*_cat` + 1 derivada), con un contrato de
calidad verificado de 7 chequeos. Con esto se cierra la preparación de datos y quedan listas las entradas
para las siguientes fases:

- **Fase 2 — Data Warehouse/Data Mart en SQLite**: las `*_cat` alimentan las dimensiones; `fact_credito`
toma `laufzeit`, `hoehe`, `rate`, `wohnzeit`, `bishkred` y `kredit`.
- **Unidad III — modelos supervisados**: `kredit_cat` como objetivo de clasificación y `hoehe` como
objetivo de regresión.

**Documentación ampliada:** `docs/analisis/etl_documentacion.md` · **Propuesta DW:**
`docs/analisis/propuesta_dw_datamart.md`
